# Session 7 — Bellman, DP, Monte Carlo, TD

**Salih Tutun, PhD** · DAT 5668 · Module 4 · Concept · Explain → practice together

## How to use this notebook

1. Read **Part I** before coding.  
2. Every stop: **explain → small cell → run → talk**.

| Cue | Meaning |
| --- | --- |
| **RUN WITH ME** | Ready code. Run. Do not edit. |
| **TRY THIS** | Type `<<< FILL`. Re-run. Check. |
| **CHANGE ME** | Change the marked value. Re-run. **PAIR 20s**. |
| **CHECK** | PASS / NOT YET |

![Bellman backup](https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/bellman.png)
![Inventory MDP](https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/mdp-inventory.png)

*All diagrams load from GitHub raw URLs.*


# Part I — Topic (explain before coding)

## Same Electronics desk — decide is now a **value**

Sessions 5–6: MDP + go/no-go memo.  
Today: the decide box is **V(s)**, not a tool call.

```text
observe inventory → decide (best order from V) → demand → reward → next inventory
```

| Idea | One-line meaning |
| --- | --- |
| **V(s)** | Expected discounted reward from stock `s` if we keep acting this way |
| **Bellman backup** | reward + γ × next V, averaged with **known** P(d) |
| **Value iteration** | Repeat backups until the table settles |
| **Monte Carlo** | Wait for episode end; assign the return |
| **TD** | Update after one step using the next state's current V |

### Lab 4 bridge — model-based vs model-free

| Approach | Needs P(d)? | When |
| --- | --- | --- |
| Value iteration (tonight) | **Yes** | Small known model / whiteboard |
| Q-learning (Session 8) | **No** | Real Electronics days as samples |

**Teach aloud:** A value ≠ this morning's profit. DP dies when P(d) is unknown or the state explodes.

**JUMP BACK** — do not open Stop 0 until I say OPEN.


---
# Part II — Practice on Electronics (`inventory_dataset_full.csv`)


## Stop 0 — Imports + settings


In [ ]:
# STOP 0a — RUN WITH ME · imports

from pathlib import Path
from collections import Counter
import csv
import statistics

import numpy as np

try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

DRIVE_DATA_FOLDER = "/content/drive/MyDrive/DAT5668/data"
PRODUCT = "Electronics"


### 0b — Find the CSV (RUN WITH ME)


In [ ]:
# STOP 0b — RUN WITH ME · course_data

def course_data(name: str) -> Path:
    # Prefer Drive, then local folders, then GitHub download (Colab-safe)
    candidates = []
    if DRIVE_DATA_FOLDER:
        candidates.append(Path(DRIVE_DATA_FOLDER) / name)
    for base in (
        Path("data"),
        Path("../../Drive-Folder-for-course/DAT5668/data"),
        Path("../Drive-Folder-for-course/DAT5668/data"),
        Path("../../../Drive-Folder-for-course/DAT5668/data"),
        Path("../../assets/data"),
        Path("assets/data"),
        Path("/content/data"),
        Path("../data"),
    ):
        candidates.append(base / name)
    for path in candidates:
        if path.exists():
            return path
    if name in {"inventory_dataset_full.csv", "inventory_dataset.csv"}:
        import urllib.request
        dest = Path("/content/data") if Path("/content").exists() else Path("data")
        dest.mkdir(parents=True, exist_ok=True)
        out = dest / name
        url = "https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/data/" + name
        print("Downloading", name, "from GitHub…")
        urllib.request.urlretrieve(url, out)
        return out
    raise FileNotFoundError("Could not find " + name + ". Mount Drive DAT5668/data or use GitHub fallback.")


## Stop 1 — Load Electronics + empirical P(d)


In [ ]:
# STOP 1a — RUN WITH ME · load Electronics

def load_electronics_desk():
    rows = []
    with course_data("inventory_dataset_full.csv").open(encoding="utf-8") as handle:
        for row in csv.DictReader(handle):
            if (row.get("Product") or "").strip() != PRODUCT:
                continue
            rows.append({
                "Date": row["Date"],
                "Current_Inventory": int(float(row["Current_Inventory"])),
                "Demand": int(float(row["Demand"])),
                "Profit": float(row.get("Profit") or 0),
                "Action_Taken": (row.get("Action_Taken") or "").strip(),
            })
    if not rows:
        raise ValueError("No Electronics rows found")
    rows.sort(key=lambda r: r["Date"])
    return rows


ROWS = load_electronics_desk()
N_DAYS = len(ROWS)
AVG_DEMAND = statistics.mean(r["Demand"] for r in ROWS)
print("Electronics days:", N_DAYS, "| avg demand:", round(AVG_DEMAND, 1))
print("CHECK load —", "PASS" if N_DAYS == 730 else "LOOK — expected 730 Electronics days")


In [ ]:
# STOP 1b — RUN WITH ME · demand bins for Bellman P(d)

demands = sorted(r["Demand"] for r in ROWS)
q1, q2 = demands[len(demands) // 3], demands[(2 * len(demands)) // 3]


def demand_bin(d: int) -> int:
    if d <= q1:
        return 0
    if d <= q2:
        return 1
    return 2


_counts = Counter(demand_bin(r["Demand"]) for r in ROWS)
DEMAND_P = {k: _counts[k] / N_DAYS for k in (0, 1, 2)}
# Board-sized demand quantities for the tiny CAPACITY MDP
DEMAND_QTY = {0: 0, 1: 1, 2: 2}
print("P(d):", {k: round(v, 3) for k, v in DEMAND_P.items()})
print("Board demand qty for bins:", DEMAND_QTY)

if plt is not None:
    fig, ax = plt.subplots(figsize=(4.2, 2.8))
    ax.bar(["0", "1", "2"], [DEMAND_P[0], DEMAND_P[1], DEMAND_P[2]], color="#2F5D50")
    ax.set_ylabel("P(d)")
    ax.set_title("Electronics empirical demand bins")
    ax.set_ylim(0, 1)
    fig.tight_layout()
    plt.show()


## Stop 2 — Tiny MDP + Bellman backup (model known)

![Bellman](https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/bellman.png)

Board MDP (readable in class):
- state stock ∈ {0,…,CAPACITY}
- action order ∈ {0,1,2}
- demand qty ∈ {0,1,2} with probabilities from Electronics `DEMAND_P`


In [ ]:
# STOP 2a — RUN WITH ME · transition + one Bellman backup

CAPACITY = 4
GAMMA = 0.95
HOLD, MARGIN, STOCKOUT = 0.2, 2.0, 1.5


def transition(stock, order, demand_bin_id):
    demand_qty = DEMAND_QTY[demand_bin_id]
    started = min(CAPACITY, stock + order)
    sales = min(started, demand_qty)
    ending = started - sales
    missed = max(0, demand_qty - started)
    pay = MARGIN * sales - HOLD * ending - STOCKOUT * missed
    return ending, pay


def bellman_backup(values, stock, order):
    # E_d [ r + gamma * V(s') ] with known P(d)
    total = 0.0
    for d, prob in DEMAND_P.items():
        ending, pay = transition(stock, order, d)
        total += prob * (pay + GAMMA * values[ending])
    return total


# Worked example on the board: V=0 everywhere, stock=0, order=2
v0 = np.zeros(CAPACITY + 1)
print("Backup at stock=0, order=2 when V=0 everywhere:", round(bellman_backup(v0, 0, 2), 3))
print("(That is mostly expected immediate reward — next V is still zero.)")


### 2b — CHANGE ME · γ on value iteration

**PAIR 20s.** Change `GAMMA`. Re-run. Does empty-shelf still want to order?


In [ ]:
# STOP 2b — CHANGE ME · value iteration

GAMMA = 0.95  # <<< CHANGE ME: try 0.5 or 0.99

values = np.zeros(CAPACITY + 1)
history = []
for sweep in range(40):
    new_values = np.zeros_like(values)
    for stock in range(CAPACITY + 1):
        new_values[stock] = max(bellman_backup(values, stock, a) for a in range(3))
    history.append(float(np.max(np.abs(new_values - values))))
    values = new_values

policy = [int(np.argmax([bellman_backup(values, s, a) for a in range(3)])) for s in range(CAPACITY + 1)]
print("GAMMA", GAMMA)
print("V(s):", [round(v, 2) for v in values])
print("best order by stock:", policy)

if plt is not None:
    fig, axes = plt.subplots(1, 3, figsize=(9.5, 2.8))
    axes[0].bar(range(CAPACITY + 1), values, color="#2F5D50")
    axes[0].set_title("V(s)")
    axes[0].set_xlabel("stock")
    axes[1].bar(range(CAPACITY + 1), policy, color="#1F4E79")
    axes[1].set_yticks([0, 1, 2])
    axes[1].set_title("greedy order")
    axes[1].set_xlabel("stock")
    axes[2].plot(history, color="#C45C26")
    axes[2].set_title("max |ΔV| per sweep")
    axes[2].set_xlabel("sweep")
    fig.tight_layout()
    plt.show()


### 2c — TRY THIS · empty shelf

What order does value iteration place at stock = 0?


In [ ]:
# STOP 2c — TRY THIS

order_at_empty = None  # <<< FILL: 0, 1, or 2
learned = int(np.argmax([bellman_backup(values, 0, a) for a in range(3)]))
print("You said", order_at_empty, "| VI says", learned)
if order_at_empty is None:
    print("NOT YET")
else:
    print("CHECK 2 —", "PASS" if order_at_empty == learned else "LOOK — defend your difference")


## Stop 3 — Monte Carlo vs TD (same episode)

![Q preview](https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/q-update.png)

- **MC**: wait until the end; walk returns backward.  
- **TD**: one-step target `r + γ V(s')`.  
Session 8 Q-learning = TD for **Q(s,a)** — no P(d).


In [ ]:
# STOP 3 — RUN WITH ME · one episode

def behavior_order(stock):
    return 1 if stock < 2 else 0


episode = []
stock = 1
for d in (2, 0, 1, 2):  # fixed demand bins so the board matches
    a = behavior_order(stock)
    ending, pay = transition(stock, a, d)
    episode.append((stock, a, pay, ending))
    stock = ending

# Monte Carlo backward returns
mc = []
g = 0.0
for stock_t, _a, pay, _e in reversed(episode):
    g = pay + GAMMA * g
    mc.append((stock_t, g))
mc.reverse()

stock0, _a0, pay0, next0 = episode[0]
td_target = pay0 + GAMMA * values[next0]

print("episode (s, a, r, s'):", [(s, a, round(r, 2), sp) for s, a, r, sp in episode])
print("MC returns:", [(s, round(g, 2)) for s, g in mc])
print(f"TD target for s={stock0}: {td_target:.2f}  vs current V={values[stock0]:.2f}")

if plt is not None:
    fig, ax = plt.subplots(figsize=(5.5, 2.8))
    xs = range(len(mc))
    ax.bar([x - 0.15 for x in xs], [g for _, g in mc], width=0.3, label="MC return", color="#2F5D50")
    ax.bar([0 + 0.15], [td_target], width=0.3, label="TD target (step 0)", color="#C45C26")
    ax.set_xticks(list(xs), [f"s={s}" for s, _ in mc])
    ax.set_title("MC returns vs one TD target")
    ax.legend(fontsize=8)
    fig.tight_layout()
    plt.show()


## Stop 4 — When DP is the wrong tool

You cannot run Bellman if P(d) is unknown, every SKU is a state, or the world changes.  
**Session 8 / Lab 4** trades the table for samples (ε-greedy + Q-update). Samples cost money.

| Tonight | Next class |
| --- | --- |
| DP with known Electronics P(d) | Q-table on Electronics days — **no** P(d) |


## Stop 5 — Framework line (TRY THIS)


In [ ]:
# STOP 5 — TRY THIS

framework_family = ""  # <<< FILL: LLM / RL / hybrid
framework_line = ""    # <<< FILL: one sentence you will keep
print("Family:", framework_family or "(FILL)")
print("I will keep:", framework_line or "(FILL)")
print("CHECK 5 —", "PASS" if framework_family and framework_line else "NOT YET")


## Raise a hand — Session 7 complete

1. [ ] Can define V(s) in one sentence  
2. [ ] Value-iteration plots discussed; empty shelf order checked  
3. [ ] Can contrast MC vs TD  
4. [ ] Can say why Session 8 drops P(d)  
